# 🏠 Proyecto: SmartHome Systems (Ruta Crítica - CPM)

Este notebook implementa el **Método de la Ruta Crítica (CPM)** para planificar la integración de módulos de un sistema domótico.

**Objetivo:** Identificar la duración mínima del proyecto y las actividades críticas que no pueden sufrir retrasos.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Configuración visual
%matplotlib inline

## 1. Definición del Proyecto

Modelamos las actividades A-F con sus duraciones y dependencias según el caso de estudio.

In [ ]:
def definir_proyecto_smarthome():
    # Estructura: {ID: {'dur': días, 'pred': [predecesores]}}
    # Datos basados en la tabla del Caso de Estudio U3 AC
    return {
        'A': {'dur': 4, 'pred': [], 'desc': 'Diseño Protocolo'},
        'B': {'dur': 6, 'pred': ['A'], 'desc': 'Módulo Conversión'},
        'C': {'dur': 3, 'pred': ['A'], 'desc': 'Configuración IP'},
        'D': {'dur': 5, 'pred': ['B', 'C'], 'desc': 'Pruebas Compatibilidad'},
        'E': {'dur': 4, 'pred': ['C'], 'desc': 'Módulo Seguridad'},
        'F': {'dur': 3, 'pred': ['D', 'E'], 'desc': 'Integración Final'}
    }

actividades = definir_proyecto_smarthome()

## 2. Algoritmo CPM (Cálculo de Tiempos)

Calculamos los tiempos tempranos (ES, EF), tardíos (LS, LF) y la holgura para cada actividad.

In [ ]:
def calcular_ruta_critica(actividades):
    # --- Paso Adelante (Forward Pass) ---
    # Calcular ES (Early Start) y EF (Early Finish)
    # Iteramos asegurando que los predecesores estén listos
    cola_proceso = [k for k, v in actividades.items() if not v['pred']]
    procesados = []

    while len(procesados) < len(actividades):
        for id_act in cola_proceso:
            if id_act in procesados: continue
            
            datos = actividades[id_act]
            # Verificar si todos los predecesores ya fueron procesados
            preds_listos = all([p in procesados for p in datos['pred']])
            
            if preds_listos:
                if not datos['pred']:
                    datos['ES'] = 0
                else:
                    datos['ES'] = max([actividades[p]['EF'] for p in datos['pred']])
                
                datos['EF'] = datos['ES'] + datos['dur']
                procesados.append(id_act)
        
        # Añadir sucesores a la cola
        for id_act in procesados:
             for k, v in actividades.items():
                if id_act in v['pred'] and k not in procesados and k not in cola_proceso:
                    cola_proceso.append(k)

    duracion_proyecto = max([a['EF'] for a in actividades.values()])

    # --- Paso Atrás (Backward Pass) ---
    # Calcular LS (Late Start), LF (Late Finish) y Holgura
    # Crear mapa de sucesores
    sucesores = {k: [] for k in actividades}
    for id_act, datos in actividades.items():
        for p in datos['pred']:
            sucesores[p].append(id_act)

    # Procesar en orden inverso (desde el final)
    orden_inverso = procesados[::-1]

    for id_act in orden_inverso:
        datos = actividades[id_act]
        if not sucesores[id_act]:
            datos['LF'] = duracion_proyecto
        else:
            datos['LF'] = min([actividades[s]['LS'] for s in sucesores[id_act]])
        
        datos['LS'] = datos['LF'] - datos['dur']
        datos['Holgura'] = datos['LS'] - datos['ES']
        datos['Critica'] = 'SÍ' if datos['Holgura'] == 0 else 'No'
    
    return duracion_proyecto

duracion = calcular_ruta_critica(actividades)
print(f"Duración Total del Proyecto: {duracion} días")

# Mostrar tabla de resultados
print(f"{'Act':<5} {'Desc':<20} {'Dur':<5} {'ES':<5} {'EF':<5} {'LS':<5} {'LF':<5} {'Holg':<5} {'Crítica'}")
print("-"*80)
for id_act, d in actividades.items():
    print(f"{id_act:<5} {d['desc']:<20} {d['dur']:<5} {d['ES']:<5} {d['EF']:<5} {d['LS']:<5} {d['LF']:<5} {d['Holg']:<5} {d['Critica']}")

## 3. Visualización de la Ruta Crítica

Generamos el diagrama de red resaltando las actividades críticas en rojo.

In [ ]:
def visualizar_proyecto(actividades):
    G = nx.DiGraph()
    for act, datos in actividades.items():
        # Etiqueta con nombre, duración y holgura
        lbl = f"{act}\n{datos['dur']}d\nH={datos['Holgura']}"
        G.add_node(act, label=lbl)
        for p in datos['pred']:
            G.add_edge(p, act)

    # Layout jerárquico manual para mayor claridad
    pos = {
        'A': (0, 5),
        'B': (3, 7), 'C': (3, 3),
        'D': (6, 7), 'E': (6, 3),
        'F': (9, 5)
    }

    plt.figure(figsize=(12, 6))
    plt.title("Ruta Crítica: Proyecto SmartHome", fontsize=14)

    # Identificar elementos críticos
    nodos_criticos = [n for n in actividades if actividades[n]['Critica'] == 'SÍ']
    aristas_criticas = []
    for u, v in G.edges():
        if u in nodos_criticos and v in nodos_criticos:
            # La arista es crítica si conecta dos tareas críticas sin holgura temporal intermedia
            if actividades[u]['EF'] == actividades[v]['ES']:
                aristas_criticas.append((u, v))

    # Dibujar grafo base
    nx.draw_networkx_edges(G, pos, edge_color='lightgray', arrowsize=20)
    nx.draw_networkx_nodes(G, pos, node_size=2000, node_color='lightblue', edgecolors='gray')

    # Resaltar ruta crítica
    nx.draw_networkx_nodes(G, pos, nodelist=nodos_criticos, node_color='#FF6F61', node_size=2000, edgecolors='red')
    nx.draw_networkx_edges(G, pos, edgelist=aristas_criticas, edge_color='red', width=2.5, arrowsize=25)

    # Etiquetas
    labels = nx.get_node_attributes(G, 'label')
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=10, font_weight='bold')

    plt.axis('off')
    plt.show()

visualizar_proyecto(actividades)